# Cell-level contrastive embeddings: extraction, UMAP, and sub-class discovery

This notebook has two independent parts:

1. **Part 1 - Extract.** Run the trained panoptic model over a folder of
   images, saving for every image a full-resolution **cell mask** and, for
   every detected cell, its pooled **contrastive feature embedding** plus
   phenotype label and bookkeeping info (bbox, centroid, source image).
   Needs a GPU, detectron2, and the trained checkpoint.

2. **Part 2 - Analyze.** Load the saved `cells.npz` + mask files, project
   the embeddings with **UMAP** (or t-SNE), look for **sub-classes** within
   the existing phenotype calls with KMeans, and make figures that pair the
   **original image** with the **embedding distribution**.

Part 2 only needs `contrast_learning.py`, numpy/pandas/sklearn/umap-learn/
matplotlib, the saved `cells.npz` + `masks/` folder, and the original
images. **No model, GPU, or detectron2 required** - this is what you
re-run on a different machine after copying over the `output_folder`.

If you only want to re-analyze previously extracted results, skip to
**Part 2** below and point `OUTPUT_FOLDER` / `IMAGE_FOLDER` at your data.

## Part 1 - Extract per-image masks and per-cell embeddings

Requires GPU + detectron2 + the trained checkpoint.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("./detectron2"))
sys.path.insert(0, os.path.abspath("."))
sys.path.insert(0, os.path.abspath("./projects/Panoptic-DeepLab/"))

# Only needed if your dataset is registered through DETECTRON2_DATASETS;
# harmless to leave set even when you are only running inference.
os.environ.setdefault("DETECTRON2_DATASETS", "/path/to/datasets/")

In [ ]:
from detectron2.config import get_cfg
from detectron2.engine import default_setup
from detectron2.projects.panoptic_deeplab import (
    PanopticDeeplabDatasetMapper,
    add_panoptic_deeplab_config,
)

from prediction import Predictor

### Configuration -- edit these for your machine

Everything below is the only thing that should need to change when you
move this notebook to a different machine.

In [ ]:
# --- Part 1 inputs -----------------------------------------------------
CONFIG_FILE = "./projects/Panoptic-DeepLab/configs/yeast_panoptics/config.yaml"
MODEL_WEIGHTS = "/path/to/model_final.pth"
DEVICE = "cuda:0"                 # "cpu" also works, just much slower
SCORE_THRESH_TEST = 0.3

IMAGE_FOLDER = "./images/"        # folder of input images to process
TARGET_LAYER = "ins_embed_head.contrast_head"

# --- Part 1 outputs / extraction thresholds -----------------------------
OUTPUT_FOLDER = "./cell_embeddings"   # cells.npz + masks/ are written here
MIN_ORIGINAL_PIXELS = 20              # minimum cell size, full resolution
MIN_FEATURE_PIXELS = 2                # minimum cell size, feature resolution
CATEGORY_ID_OFFSET = 0                # set to 1 if categories are 0-indexed
OVERWRITE_EXISTING = False            # set True to redo a previous extraction

# --- Mask smoothing -------------------------------------------------------
FILL_HOLES = True          # fill small gaps/holes inside a cell's mask
HOLE_AREA_THRESHOLD = None # None = fill every hole; set an int (pixels) to
                            # bound it instead, e.g. if a phenotype can
                            # legitimately contain another instance inside
                            # it (spores inside an ascus)
REMOVE_BORDER_CELLS = True # drop cells truncated by the image border
BORDER_WIDTH = 1           # how many edge pixels count as "the border"

In [ ]:
cfg = get_cfg()
add_panoptic_deeplab_config(cfg)
cfg.merge_from_file(CONFIG_FILE)

cfg.MODEL.DEVICE = DEVICE
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESH_TEST
cfg.MODEL.WEIGHTS = os.path.abspath(MODEL_WEIGHTS)

predictor = Predictor(cfg)

In [ ]:
from contrast_learning import extract_cell_embeddings_from_folder

cells_data = extract_cell_embeddings_from_folder(
    predictor=predictor,
    image_folder=IMAGE_FOLDER,
    target_layer=TARGET_LAYER,
    output_folder=OUTPUT_FOLDER,
    min_original_pixels=MIN_ORIGINAL_PIXELS,
    min_feature_pixels=MIN_FEATURE_PIXELS,
    category_id_offset=CATEGORY_ID_OFFSET,
    overwrite=OVERWRITE_EXISTING,
    fill_holes=FILL_HOLES,
    hole_area_threshold=HOLE_AREA_THRESHOLD,
    remove_border_cells=REMOVE_BORDER_CELLS,
    border_width=BORDER_WIDTH,
)

This wrote, under `OUTPUT_FOLDER`:

- `cells.npz` - one row per cell: embedding, phenotype label, segment id,
  source image name/path, mask file, pixel count, bbox, centroid, and
  source image size.
- `masks/<image_name>.cells.npy` - one full-resolution integer mask per
  input image (`0` = background, otherwise the cell's unique `cell_id`).

Each cell's mask was smoothed before saving: interior holes filled and
cells touching the image border dropped (see `postprocess_cell_mask()` in
`contrast_learning.py`), so pixel counts, bboxes, and embeddings all
reflect the cleaned-up mask, not the raw predictor output.

Everything from here on only needs those two outputs plus the original
images, so it is safe to copy `OUTPUT_FOLDER` (and the images) to another
machine and continue from Part 2.

## Part 2 - Analyze: UMAP distribution and sub-class discovery

Re-runnable on any machine with `contrast_learning.py` and the usual
scientific-Python stack, plus `umap-learn` (`pip install umap-learn`).
If you are starting fresh in a new session, just set `OUTPUT_FOLDER` and
`IMAGE_FOLDER` above (or re-run the cell below) and start here -- no need
to reload the model.

In [ ]:
import numpy as np
import pandas as pd

from contrast_learning import (
    PHENOTYPE_COLORS,
    PHENOTYPE_NAMES,
    add_projection_and_clusters,
    cluster_embeddings,
    evaluate_kmeans_k,
    load_cell_table,
    plot_cell_gallery,
    plot_image_with_cells,
    plot_kmeans_k_sweep,
    plot_umap,
    plot_umap_with_image_panel,
)

table, embeddings = load_cell_table(OUTPUT_FOLDER)

print(f"{len(table)} cells from {table['image_name'].nunique()} images")
table["phenotype_name"].value_counts()

### 2.1 - UMAP projection of all cells, colored by phenotype

In [ ]:
table = add_projection_and_clusters(table, embeddings, method="umap")
umap_coordinates = table[["umap_1", "umap_2"]].to_numpy()

plot_umap(
    coordinates=umap_coordinates,
    labels=table["label_id"].to_numpy(),
    label_names=PHENOTYPE_NAMES,
    label_colors=PHENOTYPE_COLORS,
    cutoff_number=1000,
    output_path=f"{OUTPUT_FOLDER}/umap_by_phenotype.png",
)

### 2.2 - How many sub-classes are there?

Sweep KMeans cluster counts and look at the elbow (inertia) and the
silhouette score (higher = better separated) to pick a reasonable `k`.
Clustering runs on the same L2-normalized embeddings used for UMAP, not
on the 2D projection, so it is not biased by UMAP's local distortions.

In [ ]:
k_sweep = evaluate_kmeans_k(embeddings, k_values=range(2, 15))
plot_kmeans_k_sweep(k_sweep, output_path=f"{OUTPUT_FOLDER}/kmeans_k_sweep.png")
k_sweep

In [ ]:
# Pick k from the sweep above, then assign sub-class clusters.
N_CLUSTERS = 8

table = add_projection_and_clusters(
    table, embeddings, method="umap", n_clusters=N_CLUSTERS,
)

plot_umap(
    coordinates=umap_coordinates,
    labels=table["cluster_id"].to_numpy(),
    cutoff_number=1000,
    output_path=f"{OUTPUT_FOLDER}/umap_by_cluster.png",
)

Cross-tabulate clusters against the existing phenotype calls: a cluster
that splits cleanly across phenotypes is probably a real sub-class; one
that matches an existing phenotype 1:1 probably is not.

In [ ]:
pd.crosstab(table["cluster_id"], table["phenotype_name"])

In [ ]:
# Save the combined per-cell table (metadata + UMAP coords + cluster id).
# Embeddings stay in cells.npz; join on "cell_id" if you need them too.
table.to_csv(f"{OUTPUT_FOLDER}/cell_table.csv", index=False)

### 2.3 - Inspect a sub-cluster: original image crops

Pick a cluster (or a phenotype) of interest and look at the actual cells,
not just the embedding -- this is where you confirm whether a UMAP
sub-cluster corresponds to a real, visually distinct sub-class.

In [ ]:
CLUSTER_OF_INTEREST = 0

selected_cell_ids = (
    table.loc[table["cluster_id"] == CLUSTER_OF_INTEREST, "cell_id"]
    .sample(n=min(24, (table["cluster_id"] == CLUSTER_OF_INTEREST).sum()), random_state=0)
)

plot_cell_gallery(
    table=table,
    cell_ids=selected_cell_ids,
    output_folder=OUTPUT_FOLDER,
    image_folder=IMAGE_FOLDER,
    figure_path=f"{OUTPUT_FOLDER}/cluster_{CLUSTER_OF_INTEREST}_gallery.png",
)

### 2.4 - Linked figure: one image + where its cells sit in the distribution

Left panel: a full source image with its cell masks colored by cluster.
Right panel: the full UMAP scatter with that image's cells outlined, so
you can see at a glance which part(s) of the distribution it populates.

In [ ]:
IMAGE_OF_INTEREST = table["image_name"].iloc[0]

plot_umap_with_image_panel(
    coordinates=umap_coordinates,
    table=table,
    image_name=IMAGE_OF_INTEREST,
    output_folder=OUTPUT_FOLDER,
    image_folder=IMAGE_FOLDER,
    color_by="cluster_id",
    figure_path=f"{OUTPUT_FOLDER}/{IMAGE_OF_INTEREST}_umap_panel.png",
)

### Resuming later / on another machine

Everything in Part 2 only needs:

- `OUTPUT_FOLDER/cells.npz` and `OUTPUT_FOLDER/masks/`
- the original images (or just re-point `IMAGE_FOLDER`)
- `OUTPUT_FOLDER/cell_table.csv` (optional convenience export from 2.2)

Re-run from the `import` cell at the top of Part 2 -- `load_cell_table()`
rebuilds everything `add_projection_and_clusters()` and the plotting
helpers need, with no model or GPU involved.